1. The Architecture Blueprint
    
    - Before writing the code, let's map out exactly what we are connecting:
        - Source Input $\rightarrow$ Source Embeddings + Positional Encoding $\rightarrow$ Encoder Stack.
        - Target Input (shifted right during training) $\rightarrow$ Target Embeddings + Positional Encoding $\rightarrow$ Decoder Stack.
        - Encoder Output (Key and Value matrices) $\rightarrow$ Passed into every layer of the Decoder Stack (Cross-Attention).
        - Decoder Output $\rightarrow$ Linear projection layer $\rightarrow$ Logits (Vocabulary size).

---
##### Basic Transformer

In [2]:
import math
import time
import copy
import random
from dataclasses import dataclass
from collections import Counter

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)
PAD_IDX = 0
BOS_IDX = 1
EOS_IDX = 2
UNK_IDX = 3

class FeedForward(nn.Module):
    def __init__(self, d_model=512, d_ff=2048, dropout=0.1):
        super(FeedForward, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(d_ff, d_model)
        )
    
    def forward(self, x):
        return self.net(x)

class EncoderLayer(nn.Module):
    def __init__(self, d_model=512, n_heads=8, d_ff=2048, dropout=0.1):
        super(EncoderLayer, self).__init__()

        self.self_attn = nn.MultiheadAttention(
            d_model, n_heads, dropout=dropout, batch_first=True
        )
        self.ffn = FeedForward(d_model, d_ff, dropout)

        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, x, src_key_padding_mask=None):
        residual = x
        x_norm = self.norm1(x)
        attn_out, _ = self.self_attn(
            x_norm, x_norm, x_norm,
            key_padding_mask=src_key_padding_mask
        )
        x = residual + self.dropout(attn_out)

        residual = x
        x_norm = self.norm2(x)
        ffn_out = self.ffn(x_norm)
        x = residual + self.dropout(ffn_out)

        return x

class DecoderLayer(nn.Module):
    def __init__(self, d_model=512, n_heads=8, d_ff=2048, dropout=0.1):
        super(DecoderLayer, self).__init__()

        self.self_attn = nn.MultiheadAttention(
            d_model, n_heads, dropout=dropout, batch_first=True
        )
        self.cross_attn = nn.MultiheadAttention(
            d_model, n_heads, dropout=dropout, batch_first=True
        )
        self.ffn = FeedForward(d_model, d_ff, dropout)

        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, x, memory, tgt_mask=None, tgt_key_padding_mask=None, memory_key_padding_mask=None):
        residual = x
        x_norm = self.norm1(x)
        attn_out, _ = self.self_attn(
            x_norm, x_norm, x_norm,
            attn_mask=tgt_mask,
            key_padding_mask=tgt_key_padding_mask
        )
        x = residual + self.dropout(attn_out)

        residual = x
        x_norm = self.norm2(x)
        cross_attn_out, _ = self.cross_attn(
            x_norm, memory, memory,
            key_padding_mask=memory_key_padding_mask
        )
        x = residual + self.dropout(cross_attn_out)

        residual = x
        x_norm = self.norm3(x)
        ffn_out = self.ffn(x_norm)
        x = residual + self.dropout(ffn_out)

        return x

class Encoder(nn.Module):
    def __init__(self, vocab_size, d_model=512, n_heads=8, n_layers=6, d_ff=2048, dropout=0.1, max_len=5000):
        super(Encoder, self).__init__()
        self.token_embed = nn.Embedding(vocab_size, d_model, padding_idx=PAD_IDX)
        self.pos_embed = nn.Embedding(max_len, d_model)
        self.layers = nn.ModuleList([EncoderLayer(d_model, n_heads, d_ff, dropout) for _ in range(n_layers)])
        self.norm = nn.LayerNorm(d_model)
        self.d_model = d_model

    def forward(self, src, src_key_padding_mask=None):
        x = self.token_embed(src) * math.sqrt(self.d_model)
        x = x + self.pos_embed(torch.arange(src.size(1), device=src.device).unsqueeze(0))
        for layer in self.layers:
            x = layer(x, src_key_padding_mask)
        
        return self.norm(x)
    
class Decoder(nn.Module):
    def __init__(self, vocab_size, d_model=512, n_heads=8, n_layers=6, d_ff=2048, dropout=0.1, max_len=5000):
        super(Decoder, self).__init__()
        self.token_embed = nn.Embedding(vocab_size, d_model, padding_idx=PAD_IDX)
        self.pos_embed = nn.Embedding(max_len, d_model)
        self.layers = nn.ModuleList([DecoderLayer(d_model, n_heads, d_ff, dropout) for _ in range(n_layers)])
        self.norm = nn.LayerNorm(d_model)
        self.d_model = d_model
    
    def forward(self, tgt, memory, tgt_mask=None, tgt_key_padding_mask=None, memory_key_padding_mask=None):
        x = self.token_embed(tgt) * math.sqrt(self.d_model)
        x = x + self.pos_embed(torch.arange(tgt.size(1), device=tgt.device).unsqueeze(0))
        for layer in self.layers:
            x = layer(x, memory, tgt_mask, tgt_key_padding_mask, memory_key_padding_mask)
            
        return self.norm(x)
    
class Transformer(nn.Module):
    def __init__(
        self,
        src_vocab_size,
        tgt_vocab_size,
        d_model=512,
        n_heads=8,
        n_layers=6,
        d_ff=2048,
        dropout=0.1,
        max_len=5000
    ):
        super(Transformer, self).__init__()

        self.encoder = Encoder(src_vocab_size, d_model, n_heads, n_layers, d_ff, dropout, max_len)
        self.decoder = Decoder(tgt_vocab_size, d_model, n_heads, n_layers, d_ff, dropout, max_len)
        self.output_proj = nn.Linear(d_model, tgt_vocab_size)
    
    def forward(self, src, tgt, src_key_padding_mask=None, tgt_key_padding_mask=None, tgt_mask=None):
        memory = self.encoder(src, src_key_padding_mask)
        output = self.decoder(
            tgt,
            memory,
            tgt_mask=tgt_mask,
            tgt_key_padding_mask=tgt_key_padding_mask,
            memory_key_padding_mask=src_key_padding_mask
        )
        return self.output_proj(output)

def generate_square_subsequent_mask(sz, device=None):
    if device is None:
        device = torch.device("cpu")
    mask = torch.triu(torch.ones(sz, sz, device=device), diagonal=1).bool()
    return mask

def create_padding_mask(seq, pad_idx=PAD_IDX):
    return (seq == pad_idx)



Using device: cuda


---
#### English --> Japanese Translator

In [3]:
import math
import random
from collections import Counter

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, random_split
import pandas as pd

SPECIAL_TOKENS = ["<pad>", "<bos>", "<eos>", "<unk>"]

random.seed(42)
torch.manual_seed(42)

# =========================
# 2. Sample English-Japanese dataset
# =========================


file_path = "JapaneseNLP.csv"   # change filename here
df = pd.read_csv(file_path)

print(df.head())
print(df.columns)

# remove missing rows
df = df[["English", "Japan"]].dropna()

# convert to strings
df["English"] = df["English"].astype(str)
df["Japan"] = df["Japan"].astype(str)

pairs = list(zip(df["English"], df["Japan"]))

print("Total pairs:", len(pairs))
print("First pair:", pairs[0])

# =========================
# 3. Tokenizers
# =========================
def tokenize_en(text):
    # very simple word-level tokenizer
    text = text.lower().strip()
    for p in [".", ",", "?", "!", ":", ";"]:
        text = text.replace(p, f" {p} ")
    return text.split()

def tokenize_ja(text):
    # simple character-level tokenizer
    # good for small demo data, not ideal for real-world use
    return list(text.strip())

def build_vocab(sentences, tokenizer, min_freq=1):
    counter = Counter()
    for sent in sentences:
        counter.update(tokenizer(sent))
    
    stoi = {token: idx for idx, token in enumerate(SPECIAL_TOKENS)}

    for token, freq in counter.items():
        if freq >= min_freq and token not in stoi:
            stoi[token] = len(stoi)
    
    itos = {idx: token for token, idx in stoi.items()}
    return stoi, itos

src_text = [src for src, tgt in pairs]
tgt_text = [tgt for src, tgt in pairs]

src_stoi, src_itos = build_vocab(src_text, tokenize_en)
tgt_stoi, tgt_itos = build_vocab(tgt_text, tokenize_ja)

SRC_VOCAB_SIZE = len(src_stoi)
TGT_VOCAB_SIZE = len(tgt_stoi)

print("English vocab size:", SRC_VOCAB_SIZE)
print("Japanese vocab size:", TGT_VOCAB_SIZE)

def numericalize(text, stoi, tokenizer):
    return [stoi.get(token, UNK_IDX) for token in tokenizer(text)]

def encode_src(text):
    return [BOS_IDX] + numericalize(text, src_stoi, tokenize_en) + [EOS_IDX]

def encode_tgt(text):
    return [BOS_IDX] + numericalize(text, tgt_stoi, tokenize_ja) + [EOS_IDX]

def decode_tgt_ids(ids):
    tokens = []
    for idx in ids:
        if idx == EOS_IDX:
            break
        if idx in [PAD_IDX, BOS_IDX]:
            continue
        tokens.append(tgt_itos.get(idx, "<unk>"))
    return "".join(tokens)



class TranslationDataset(Dataset):
    def __init__(self, pairs):
        self.data = []
        for src, tgt in pairs:
            src_ids = encode_src(src)
            tgt_ids = encode_tgt(tgt)
            self.data.append((src_ids, tgt_ids))

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        return self.data[idx]

def translation_collate_fn(batch):
    src_batch, tgt_batch = zip(*batch)

    src_max_len = max(len(x) for x in src_batch)
    tgt_max_len = max(len(x) for x in tgt_batch)

    padded_src = []
    padded_tgt = []

    for src_ids, tgt_ids in zip(src_batch, tgt_batch):
        padded_src.append(src_ids + [PAD_IDX] * (src_max_len - len(src_ids)))
        padded_tgt.append(tgt_ids + [PAD_IDX] * (tgt_max_len - len(tgt_ids)))

    src_tensor = torch.tensor(padded_src, dtype=torch.long)
    tgt_tensor = torch.tensor(padded_tgt, dtype=torch.long)

    return src_tensor, tgt_tensor

dataset = TranslationDataset(pairs)

train_size = int(0.7 * len(dataset))
val_size = int(0.15 * len(dataset))
test_size = len(dataset) - train_size - val_size

train_dataset, val_dataset, test_dataset = random_split(
    dataset,
    [train_size, val_size, test_size],
    generator=torch.Generator().manual_seed(42)
)

train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True, collate_fn=translation_collate_fn)
val_loader = DataLoader(val_dataset, batch_size=4, shuffle=False, collate_fn=translation_collate_fn)
test_loader = DataLoader(test_dataset, batch_size=4, shuffle=False, collate_fn=translation_collate_fn)


               English         Japan
0  Hello, how are you?   こんにちは元気ですか？
1   What is your name?     名前はなんですか？
2  Where are you from?    どこからきましたか？
3    Nice to meet you.  お会いできて嬉しいです。
4     How old are you?        何歳ですか？
Index(['English', 'Japan'], dtype='str')
Total pairs: 617
First pair: ('Hello, how are you?', 'こんにちは元気ですか？')
English vocab size: 1188
Japanese vocab size: 867


In [4]:
model = Transformer(
    src_vocab_size=SRC_VOCAB_SIZE,
    tgt_vocab_size=TGT_VOCAB_SIZE,
    d_model=256,
    n_heads=4,
    n_layers=2,
    d_ff=512,
    dropout=0.1,
    max_len=100
).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss(ignore_index=PAD_IDX)

def train_one_epoch(model, dataloader, optimizer, criterion):
    model.train()
    total_loss = 0

    for src, tgt in dataloader:
        src, tgt = src.to(device), tgt.to(device)

        optimizer.zero_grad()

        tgt_input = tgt[:, :-1]
        tgt_output = tgt[:, 1:]

        tgt_mask = generate_square_subsequent_mask(tgt_input.size(1), device=device)
        src_padding_mask = create_padding_mask(src).to(device)
        tgt_padding_mask = create_padding_mask(tgt_input).to(device)

        output = model(
            src,
            tgt_input,
            src_key_padding_mask=src_padding_mask,
            tgt_key_padding_mask=tgt_padding_mask,
            tgt_mask=tgt_mask
        )

        output = output.reshape(-1, output.size(-1))
        tgt_output = tgt_output.reshape(-1)

        loss = criterion(output, tgt_output)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
    
    return total_loss / len(dataloader)

@torch.no_grad()
def evaluate_translation(model, dataloader, criterion):
    model.eval()
    total_loss = 0

    for src, tgt in dataloader:
        src, tgt = src.to(device), tgt.to(device)

        tgt_input = tgt[:, :-1]
        tgt_output = tgt[:, 1:]

        src_padding_mask = create_padding_mask(src).to(device)
        tgt_padding_mask = create_padding_mask(tgt_input).to(device)
        tgt_mask = generate_square_subsequent_mask(tgt_input.size(1), device=device)

        logits = model(
            src,
            tgt_input,
            src_key_padding_mask=src_padding_mask,
            tgt_key_padding_mask=tgt_padding_mask,
            tgt_mask=tgt_mask
        )

        loss = criterion(
            logits.reshape(-1, logits.size(-1)),
            tgt_output.reshape(-1)
        )

        total_loss += loss.item()

    return total_loss / len(dataloader)

num_epochs = 50

for epoch in range(num_epochs):
    train_loss = train_one_epoch(model, train_loader, optimizer, criterion)
    val_loss = evaluate_translation(model, val_loader, criterion)

    print(f"Epoch {epoch+1:02d} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")

# =========================
# 12. Test loss
# =========================
test_loss = evaluate_translation(model, test_loader, criterion)
print("\nTest Loss:", round(test_loss, 4))

# =========================
# 13. Greedy decoding for inference
# =========================
@torch.no_grad()
def translate_sentence(model, sentence, max_len=50):
    model.eval()

    src_ids = encode_src(sentence)
    src_tensor = torch.tensor([src_ids], dtype=torch.long, device=device)

    src_padding_mask = create_padding_mask(src_tensor).to(device)
    memory = model.encoder(src_tensor, src_padding_mask)

    generated = [BOS_IDX]

    for _ in range(max_len):
        tgt_tensor = torch.tensor([generated], dtype=torch.long, device=device)
        tgt_padding_mask = create_padding_mask(tgt_tensor).to(device)
        tgt_mask = generate_square_subsequent_mask(tgt_tensor.size(1), device=device)

        decoder_out = model.decoder(
            tgt_tensor,
            memory,
            tgt_mask=tgt_mask,
            tgt_key_padding_mask=tgt_padding_mask,
            memory_key_padding_mask=src_padding_mask
        )

        logits = model.output_proj(decoder_out)
        next_token = logits[:, -1, :].argmax(dim=-1).item()

        generated.append(next_token)

        if next_token == EOS_IDX:
            break

    return decode_tgt_ids(generated)

# =========================
# 14. Try some translations
# =========================
examples = [
    "Hello, how are you?",
    "What is your name?",
    "I am a student.",
    "Good morning.",
    "Please help me."
]

print("\nSample translations:")
for sent in examples:
    pred = translate_sentence(model, sent)
    print(f"EN: {sent}")
    print(f"JA: {pred}")
    print("-" * 40)

Epoch 01 | Train Loss: 4.4745 | Val Loss: 3.6184
Epoch 02 | Train Loss: 3.0447 | Val Loss: 3.0936
Epoch 03 | Train Loss: 2.4814 | Val Loss: 2.8049
Epoch 04 | Train Loss: 2.0833 | Val Loss: 2.6489
Epoch 05 | Train Loss: 1.7313 | Val Loss: 2.5026
Epoch 06 | Train Loss: 1.4359 | Val Loss: 2.3956
Epoch 07 | Train Loss: 1.1766 | Val Loss: 2.3238
Epoch 08 | Train Loss: 0.9578 | Val Loss: 2.2963
Epoch 09 | Train Loss: 0.7607 | Val Loss: 2.2726
Epoch 10 | Train Loss: 0.6211 | Val Loss: 2.2838
Epoch 11 | Train Loss: 0.5296 | Val Loss: 2.2908
Epoch 12 | Train Loss: 0.4376 | Val Loss: 2.3346
Epoch 13 | Train Loss: 0.3738 | Val Loss: 2.3825
Epoch 14 | Train Loss: 0.3270 | Val Loss: 2.4353
Epoch 15 | Train Loss: 0.2815 | Val Loss: 2.4679
Epoch 16 | Train Loss: 0.2572 | Val Loss: 2.5025
Epoch 17 | Train Loss: 0.2652 | Val Loss: 2.5738
Epoch 18 | Train Loss: 0.2632 | Val Loss: 2.5875
Epoch 19 | Train Loss: 0.2494 | Val Loss: 2.5801
Epoch 20 | Train Loss: 0.2314 | Val Loss: 2.6349
Epoch 21 | Train Los

---
#### Toy Dataset

In [6]:
# ============================================
# Toy reverse-sequence dataset for full Transformer
# ============================================

class ReverseSequenceDataset(Dataset):
    def __init__(self, num_samples=1000, min_len=3, max_len=8, vocab_size=20):
        self.samples = []

        for _ in range(num_samples):
            seq_len = random.randint(min_len, max_len)
            seq = [random.randint(4, vocab_size - 1) for _ in range(seq_len)]
            reversed_seq = seq[::-1]
            self.samples.append((seq, reversed_seq))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        return self.samples[idx]


def collate_fn(batch):
    src_batch = []
    tgt_input_batch = []
    tgt_output_batch = []

    for src, tgt in batch:
        # encoder input ends with EOS
        src_tensor = torch.tensor(src + [EOS_IDX], dtype=torch.long)

        # decoder input starts with BOS
        tgt_input_tensor = torch.tensor([BOS_IDX] + tgt, dtype=torch.long)

        # expected decoder output ends with EOS
        tgt_output_tensor = torch.tensor(tgt + [EOS_IDX], dtype=torch.long)

        src_batch.append(src_tensor)
        tgt_input_batch.append(tgt_input_tensor)
        tgt_output_batch.append(tgt_output_tensor)

    src_batch = nn.utils.rnn.pad_sequence(
        src_batch, batch_first=True, padding_value=PAD_IDX
    )
    tgt_input_batch = nn.utils.rnn.pad_sequence(
        tgt_input_batch, batch_first=True, padding_value=PAD_IDX
    )
    tgt_output_batch = nn.utils.rnn.pad_sequence(
        tgt_output_batch, batch_first=True, padding_value=PAD_IDX
    )

    return src_batch, tgt_input_batch, tgt_output_batch


# ============================================
# Greedy decoding for testing
# ============================================

def greedy_decode(model, src, max_len=20):
    model.eval()

    src = src.unsqueeze(0).to(device)  # shape: (1, src_len)
    src_padding_mask = create_padding_mask(src)

    memory = model.encoder(src, src_padding_mask)

    ys = torch.tensor([[BOS_IDX]], dtype=torch.long, device=device)

    for _ in range(max_len):
        tgt_mask = generate_square_subsequent_mask(ys.size(1)).to(device)
        tgt_padding_mask = create_padding_mask(ys)

        out = model.decoder(
            ys,
            memory,
            tgt_mask=tgt_mask,
            tgt_key_padding_mask=tgt_padding_mask,
            memory_key_padding_mask=src_padding_mask
        )

        logits = model.output_proj(out)
        next_token = logits[:, -1, :].argmax(dim=-1).item()

        ys = torch.cat(
            [ys, torch.tensor([[next_token]], dtype=torch.long, device=device)],
            dim=1
        )

        if next_token == EOS_IDX:
            break

    return ys.squeeze(0).tolist()


# ============================================
# Training and evaluation
# ============================================

def evaluate(model, dataloader, criterion, vocab_size):
    model.eval()
    total_loss = 0.0

    with torch.no_grad():
        for src, tgt_input, tgt_output in dataloader:
            src = src.to(device)
            tgt_input = tgt_input.to(device)
            tgt_output = tgt_output.to(device)

            src_padding_mask = create_padding_mask(src)
            tgt_padding_mask = create_padding_mask(tgt_input)
            tgt_mask = generate_square_subsequent_mask(tgt_input.size(1)).to(device)

            logits = model(
                src,
                tgt_input,
                src_key_padding_mask=src_padding_mask,
                tgt_key_padding_mask=tgt_padding_mask,
                tgt_mask=tgt_mask
            )

            loss = criterion(
                logits.reshape(-1, vocab_size),
                tgt_output.reshape(-1)
            )

            total_loss += loss.item()

    return total_loss / len(dataloader)


# ============================================
# Main training script
# ============================================

if __name__ == "__main__":
    vocab_size = 20

    # datasets
    train_dataset = ReverseSequenceDataset(num_samples=2000, min_len=3, max_len=8, vocab_size=vocab_size)
    val_dataset = ReverseSequenceDataset(num_samples=300, min_len=3, max_len=8, vocab_size=vocab_size)
    test_dataset = ReverseSequenceDataset(num_samples=300, min_len=3, max_len=8, vocab_size=vocab_size)

    train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, collate_fn=collate_fn)
    val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, collate_fn=collate_fn)
    test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, collate_fn=collate_fn)

    # model
    model = Transformer(
        vocab_size=vocab_size,
        d_model=128,
        n_heads=4,
        n_layers=2,
        d_ff=256,
        dropout=0.1,
        max_len=50
    ).to(device)

    criterion = nn.CrossEntropyLoss(ignore_index=PAD_IDX)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

    epochs = 10
    best_val_loss = float("inf")

    for epoch in range(epochs):
        model.train()
        total_train_loss = 0.0

        for src, tgt_input, tgt_output in train_loader:
            src = src.to(device)
            tgt_input = tgt_input.to(device)
            tgt_output = tgt_output.to(device)

            src_padding_mask = create_padding_mask(src)
            tgt_padding_mask = create_padding_mask(tgt_input)
            tgt_mask = generate_square_subsequent_mask(tgt_input.size(1)).to(device)

            logits = model(
                src,
                tgt_input,
                src_key_padding_mask=src_padding_mask,
                tgt_key_padding_mask=tgt_padding_mask,
                tgt_mask=tgt_mask
            )

            loss = criterion(
                logits.reshape(-1, vocab_size),
                tgt_output.reshape(-1)
            )

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            total_train_loss += loss.item()

        avg_train_loss = total_train_loss / len(train_loader)
        avg_val_loss = evaluate(model, val_loader, criterion, vocab_size)

        print(f"Epoch {epoch+1}/{epochs}")
        print(f"Train Loss: {avg_train_loss:.4f}")
        print(f"Val Loss:   {avg_val_loss:.4f}")

        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            torch.save(model.state_dict(), "best_reverse_transformer.pt")

    # ============================================
    # Test
    # ============================================
    model.load_state_dict(torch.load("best_reverse_transformer.pt", map_location=device))
    test_loss = evaluate(model, test_loader, criterion, vocab_size)
    print(f"\nTest Loss: {test_loss:.4f}")

    # ============================================
    # Sample predictions
    # ============================================
    print("\nSample predictions:")
    for _ in range(5):
        sample_src, sample_tgt = test_dataset[random.randint(0, len(test_dataset)-1)]

        src_tensor = torch.tensor(sample_src + [EOS_IDX], dtype=torch.long)
        pred_ids = greedy_decode(model, src_tensor, max_len=20)

        # remove BOS
        pred_ids = pred_ids[1:]

        # stop at EOS
        if EOS_IDX in pred_ids:
            pred_ids = pred_ids[:pred_ids.index(EOS_IDX)]

        print(f"Input     : {sample_src}")
        print(f"Expected  : {sample_tgt}")
        print(f"Predicted : {pred_ids}")
        print("-" * 50)

c:\MIX\ASU\SEM_4\LLM_Basics\phase2\Lib\site-packages\torch\nn\modules\activation.py:1336: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  key_padding_mask = F._canonical_mask(


Epoch 1/10
Train Loss: 1.8684
Val Loss:   1.3586
Epoch 2/10
Train Loss: 1.1717
Val Loss:   0.9184
Epoch 3/10
Train Loss: 0.8250
Val Loss:   0.5910
Epoch 4/10
Train Loss: 0.6049
Val Loss:   0.4449
Epoch 5/10
Train Loss: 0.4972
Val Loss:   0.3792
Epoch 6/10
Train Loss: 0.3890
Val Loss:   0.2796
Epoch 7/10
Train Loss: 0.3377
Val Loss:   0.2678
Epoch 8/10
Train Loss: 0.3421
Val Loss:   0.2344
Epoch 9/10
Train Loss: 0.2940
Val Loss:   0.1932
Epoch 10/10
Train Loss: 0.2745
Val Loss:   0.1942

Test Loss: 0.1919

Sample predictions:
Input     : [15, 7, 5, 12]
Expected  : [12, 5, 7, 15]
Predicted : [12, 5, 7, 15]
--------------------------------------------------
Input     : [4, 5, 5, 8]
Expected  : [8, 5, 5, 4]
Predicted : [8, 5, 4]
--------------------------------------------------
Input     : [5, 12, 4, 17, 18, 9]
Expected  : [9, 18, 17, 4, 12, 5]
Predicted : [9, 18, 17, 4, 12, 5]
--------------------------------------------------
Input     : [11, 14, 15, 5, 15, 13, 9]
Expected  : [9, 13, 15